In [1]:
import pysam
import pandas as pd

In [2]:
# --- Load data ---

bamfile = '../../../../hybrid_diversity/4_read_mapping/4_hifi_reads/map-hifi/UTEX2797_hifi_v12B1_K19_filtered_q20.bam'
snpfile = '../../../../hybrid_diversity/4_read_mapping/4_hifi_reads/AIMfiles/AllAims_v2.tsv'
outfile = 'read_counts_by_chromosome.csv'
bam = pysam.AlignmentFile(bamfile, "rb")
snps = pd.read_csv(snpfile, sep="\t")


In [3]:
# Build SNP lookup for each chromosome: chrom -> {pos_0based -> (A1, A2)}
snp_lookup_by_chrom = {}
for chrom in snps["CHROM"].unique():
    snps_chrom = snps[snps["CHROM"] == chrom]
    snp_lookup_by_chrom[chrom] = {
        row["POS"] - 1: (row["A1_ALLELE"], row["A2_ALLELE"])
        for _, row in snps_chrom.iterrows()
    }


In [4]:
def genotype_read_at_snps(read, chrom):
    """
    Returns list of alleles (A1 or A2) for one read at SNP positions on the given chromosome.
    """
    if read.query_sequence is None or chrom not in snp_lookup_by_chrom:
        return []

    snp_lookup = snp_lookup_by_chrom[chrom]
    ref_to_query = {
        ref_pos: qpos
        for qpos, ref_pos in read.get_aligned_pairs()
        if qpos is not None and ref_pos is not None
    }

    alleles = []
    for pos_0based, (a1, a2) in snp_lookup.items():
        if pos_0based in ref_to_query:
            base = read.query_sequence[ref_to_query[pos_0based]]
            if base == a1:
                alleles.append("A1")
            elif base == a2:
                alleles.append("A2")

    return alleles


In [5]:
def classify_read(alleles):
    """
    Classify a read based on its allele calls:
    - 'A1': all alleles are A1
    - 'A2': all alleles are A2
    - 'mixed': mix of A1 and A2
    - 'no_calls': no genotype data
    """
    if not alleles:
        return "no_calls"

    a1_count = alleles.count("A1")
    a2_count = alleles.count("A2")

    if a2_count == 0:
        return "A1"
    elif a1_count == 0:
        return "A2"
    else:
        return "mixed"


In [6]:
# --- Process all chromosomes ---
results = []

for chrom in snp_lookup_by_chrom.keys():
    counts = {"A1": 0, "A2": 0, "mixed": 0, "no_calls": 0}
    
    for read in bam.fetch(chrom):
        if read.is_unmapped or read.reference_name != chrom:
            continue
        alleles = genotype_read_at_snps(read, chrom)
        classification = classify_read(alleles)
        counts[classification] += 1
    
    results.append({
        "chromosome": chrom,
        "A1": counts["A1"],
        "A2": counts["A2"],
        "mixed": counts["mixed"],
        "no_calls": counts["no_calls"],
        "total_reads": sum(counts.values())
    })

bam.close()

# Create results dataframe
results_df = pd.DataFrame(results)


In [7]:
# --- Display and save results ---
print("\nRead Classification Summary by Chromosome:")
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv(outfile, index=False)
print("\nResults saved to ", outfile)



Read Classification Summary by Chromosome:
chromosome    A1    A2  mixed  no_calls  total_reads
CM083856.1 13782 13064   1861      3140        31847
CM083857.1  8937  8954   1825      5997        25713
CM083858.1  8100  7487   1003      2274        18864
CM083859.1  7839  7678    938      2501        18956
CM083860.1  8213  8012   1192      3763        21180
CM083861.1  7412  7094   1020      2485        18011
CM083862.1  7096  6583    868      1688        16235
CM083863.1  5360  5199   1105      1421        13085
CM083864.1  6038 12100   1058      1588        20784
CM083865.1  5427  5431    808      1344        13010
CM083866.1  5568  5341    877      1395        13181
CM083867.1  4855  4292    637      1322        11106
CM083868.1  4153  3945    672      2581        11351
CM083869.1  4842  5068    784      1276        11970
CM083870.1  4344  4129    676      1075        10224
CM083871.1  4449  4832    732      1067        11080
CM083872.1  3916  3898    550       960         9324
CM